👉 This creates the entry point to PySpark
Think of it like:
	•	“Start Spark engine so I can process big data”


👉 These are built-in data transformation functions
	•	split → break string into parts
	•	regexp_replace → clean data using regex
	•	col → refer to a column
	•	lit → add constant value
	•	expr → write SQL-like expressions


👉 Used for Delta Lake operations (like merge/upsert)

💡 Important: This is what enables SCD Type 1 behavior.


👉 Starts Spark session (or uses existing one)


🔹 2. TRANSFORMATION FUNCTION

👉 You are defining a function that cleans and structures raw data

💡 Real-world:
This is your Bronze → Silver transformation



 3. SCD TYPE 1 FUNCTION

👉 Function to merge new data into existing table

💡 This is used in:
	•	Incremental loads
	•	Data warehouse updates

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, regexp_replace, col, lit, expr
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()


########################################
########--TRANSFORMATIONS----##########
########################################

def transform_customer(df):
    df = df.withColumn("name_parts", split(col("customer_name"), ",")) \
        .withColumn("first_name", expr("get(name_parts, 1)")) \
        .withColumn("last_name", expr("get(name_parts, 0)")) \
        .drop("name_parts")\
        .withColumn("postcode", regexp_replace(col("postcode"), r"\.0$", ""))\
        .withColumn("country", lit("USA"))\                      #lit will add constant value
        .drop("customer_name", "file_path")

    df = df.dropDuplicates(["customer_id"])


    df = df.select(
        'customer_id', 'first_name',
        'last_name', 
        'tax_id',
        'tax_code',
        'state',
        'city',
        'postcode',
        'street',
        'number',
        'unit',
        'region',
        'district',
        'country',
        'lon',
        'lat',
        'ship_to_address',
        'valid_from',
        'valid_to',
        'units_purchased',
        'loyalty_segment',
        'last_update_ts'
        )

    return df

########################################
########--SCD-1 IMPLEMENTATION----##########
########################################

def scd_merge_table(spark, source_table, target_table, business_key): 
     # Take new data (source) and merge into existing table (target 

    if not spark.catalog.tableExists(target_table):                                    #check if table exists
        print("First Load: Creating Silver Table", target_table)
        source_table.write.format("delta").mode("overwrite").saveAsTable(target_table)
        #creates table (Initial load , {full load})
        print("Table Created")

    else:
        print("Incremental Load: Performing SCD Type 1 Merge")

        delta_table = DeltaTable.forName(spark, target_table)   #Load existing table as  delta table (need for merger) 

        merge_condition = " AND ".join(
            [f"target.{col} = source.{col}" for col in business_key]     #for us  (target.customer_id = source.customer_id)
        )                                                                #this define show much record matches


        delta_table.alias("target").merge(source_table.alias("source"),    #join source + target
                                        merge_condition
                                        ).whenMatchedUpdateAll()\          # if record exists update all
                                            .whenNotMatchedInsertAll()\    # if new INSERT
                                            .execute()                     # runs the merge
        print("Merge Successfully Completed")

# this is SCD1 ,eans old data is overwritten and history is kept



########################################
########--MAIN LOGIC----##########
########################################

source_table = "ecommerce_analytics.bronze.customers"
target_table = "ecommerce_analytics.silver.customers"
business_key = ["customer_id"]              # Unique key for merge
df = spark.read.table(source_table)         # read data from bronze

source_table = transform_customer(df)       # apply transformations
scd_merge_table(spark, source_table, target_table, business_key)    #merge into silver


     


In [0]:
df.display()